# **1. Metodologia**

Este notebook transforma as disciplinas únicas em medidas e arquivos para a visualização.

| pergunta | medida principal | visão |
|---|---|---|
| Q1 · agrupamentos | BERTopic e rede de vizinhanças | mapa e nuvens de palavras |
| Q2 · semelhança a uma referência | cosseno entre embeddings | beeswarm por tópico |
| Q3 · organização dos cursos | requisitos e carga horária por tópico | grade e treemap |

Entradas: `data/processed/subjects.csv` e `subject_requisites.csv`. Saída web: `html/data/disciplinas.js`.

In [1]:
from __future__ import annotations

import hashlib
import json
import logging
import os
import re
import time
import unicodedata
import warnings
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
import networkx as nx
from bertopic import BERTopic
from bertopic.representation import MaximalMarginalRelevance
from hdbscan import HDBSCAN
from IPython.display import display
from scipy.cluster.hierarchy import leaves_list, linkage
from scipy.spatial.distance import pdist
from google import genai
from google.genai import types
from tqdm.auto import tqdm
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics import adjusted_mutual_info_score, normalized_mutual_info_score
from umap import UMAP

/Users/gabrielcarbinatto/Documents/GitHub/study-icmc/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


W1127 10:54:25.572000 520 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


### **1.1. Parâmetros e entradas**

O bloco define modelo, semente, tamanho mínimo dos tópicos e regras da rede. Em seguida carrega as tabelas processadas e a auditoria da coleta.

In [2]:
warnings.filterwarnings("ignore")
os.environ.setdefault("TORCH_CPP_LOG_LEVEL", "ERROR")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
for noisy in ("BERTopic", "torch.distributed.elastic.multiprocessing.redirects"):
    logging.getLogger(noisy).setLevel(logging.ERROR)

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
RAW_DATA = ROOT / "data" / "raw"
PROCESSED = ROOT / "data" / "processed"
SITE_DATA = ROOT / "html" / "data"

MODEL = "gemini-embedding-001"
OUTPUT_DIMENSIONALITY = 768
TASK_TYPE = "SEMANTIC_SIMILARITY"
EMBED_BATCH_SIZE = 16
EMBED_BATCH_DELAY_SECONDS = 15
SEED = 7
MIN_TOPIC = 8
K = 4
FLOOR_PERCENTILE = 85
REFRESH_EMBEDDINGS = False

subjects = pd.read_csv(PROCESSED / "subjects.csv", dtype={"codes": str}).fillna({"teachers": "", "ementa": "", "objectives": "", "program": ""})
pairs = pd.read_csv(PROCESSED / "subject_requisites.csv")
disciplines = pd.read_csv(PROCESSED / "disciplines.csv", dtype=str).fillna("")
collection = pd.read_csv(RAW_DATA / "collection_audit.csv").set_index("metric")["value"]
n = len(subjects)
print(f"{n} disciplinas únicas, {len(pairs)} pares de requisito")

528 disciplinas únicas, 702 pares de requisito


## **2. Embeddings**

O modelo `gemini-embedding-001` representa os textos com vetores de 768 dimensões, definidos por `output_dimensionality=768`. O tipo de tarefa `SEMANTIC_SIMILARITY` orienta a comparação entre ementas. Como a saída reduzida não vem normalizada, cada vetor é dividido por sua norma antes do cálculo das similaridades. O modelo aceita até 2.048 tokens por entrada. Somente o texto de ementa, objetivos e conteúdo programático é enviado, sem o nome da disciplina.

O cache em `data/processed/embeddings.npz` registra o modelo, os parâmetros e os textos de entrada. Ele é salvo a cada lote para permitir a retomada da coleta. Uma mudança nesses dados invalida o cache, e `REFRESH_EMBEDDINGS = True` força o recálculo. A chave é lida de `GEMINI_API_KEY` ou do arquivo `.env` local, que não é versionado.

In [3]:
def gemini_api_key() -> str:
    key = os.environ.get("GEMINI_API_KEY") or os.environ.get("GOOGLE_API_KEY")
    if key:
        return key
    env_file = ROOT / ".env"
    if env_file.exists():
        for line in env_file.read_text(encoding="utf-8").splitlines():
            name, separator, value = line.partition("=")
            if separator and name.strip().removeprefix("export ").strip() in {"GEMINI_API_KEY", "GOOGLE_API_KEY"}:
                key = value.strip().strip('"').strip("'")
                if key:
                    return key
    raise RuntimeError("Defina GEMINI_API_KEY no ambiente ou no arquivo .env antes de gerar os embeddings.")


texts = subjects["body_all"].tolist()
signature = hashlib.sha1("\n".join([MODEL, str(OUTPUT_DIMENSIONALITY), TASK_TYPE, *subjects["subject_id"], *texts]).encode()).hexdigest()
cache = PROCESSED / "embeddings.npz"
E = np.zeros((n, OUTPUT_DIMENSIONALITY), dtype=np.float32)
completed = np.zeros(n, dtype=bool)
if cache.exists() and not REFRESH_EMBEDDINGS:
    with np.load(cache, allow_pickle=False) as stored:
        if ("signature" in stored and str(stored["signature"]) == signature
                and stored["vectors"].shape == E.shape):
            E[:] = stored["vectors"]
            completed[:] = stored["completed"] if "completed" in stored else True

if not completed.all():
    client = genai.Client(api_key=gemini_api_key())
    last_request = 0.0
    try:
        for start in tqdm(range(0, n, EMBED_BATCH_SIZE), desc="Embeddings Gemini"):
            batch = [i for i in range(start, min(start + EMBED_BATCH_SIZE, n)) if not completed[i]]
            if not batch:
                continue
            for attempt in range(5):
                try:
                    time.sleep(max(0.0, EMBED_BATCH_DELAY_SECONDS - (time.monotonic() - last_request)))
                    last_request = time.monotonic()
                    response = client.models.embed_content(
                        model=MODEL,
                        contents=[texts[i] for i in batch],
                        config=types.EmbedContentConfig(
                            task_type=TASK_TYPE,
                            output_dimensionality=OUTPUT_DIMENSIONALITY,
                        ),
                    )
                    vectors = np.asarray([item.values for item in response.embeddings], dtype=np.float32)
                    if vectors.shape != (len(batch), OUTPUT_DIMENSIONALITY):
                        raise ValueError(f"Formato inesperado dos embeddings: {vectors.shape}")
                    norms = np.linalg.norm(vectors, axis=1, keepdims=True)
                    if not np.isfinite(vectors).all() or np.any(norms == 0):
                        raise ValueError("Embeddings inválidos retornados pela API")
                    E[batch] = vectors / norms
                    completed[batch] = True
                    np.savez_compressed(cache, ids=subjects["subject_id"].to_numpy(), vectors=E,
                                        completed=completed, signature=signature)
                    break
                except Exception as error:
                    if getattr(error, "code", None) not in {429, 500, 502, 503, 504} or attempt == 4:
                        raise
                    retry = re.search(r"Please retry in ([\d.]+)s", str(error))
                    time.sleep(max(float(retry.group(1)) + 2 if retry else 0, min(60, 3 * 2 ** attempt)))
    finally:
        client.close()

assert completed.all() and E.shape == (n, OUTPUT_DIMENSIONALITY)
print(f"embeddings: {E.shape} · modelo: {MODEL} · tarefa: {TASK_TYPE}")

embeddings: (528, 768) · modelo: gemini-embedding-001 · tarefa: SEMANTIC_SIMILARITY


## **3. Semelhança**

A semelhança é o cosseno entre embeddings normalizados:

$$
s(a,b)=\frac{\mathbf{e}_a\cdot\mathbf{e}_b}{\lVert\mathbf{e}_a\rVert\,\lVert\mathbf{e}_b\rVert}.
$$

A distribuição de todos os pares fornece a referência para interpretar vizinhanças e requisitos. Os limiares são definidos por percentis dessa distribuição, e não por valores fixos, porque a escala do cosseno depende do modelo. O percentil 85 é o limiar da rede, e o percentil 90 é o limiar da Q2: uma disciplina só entra na comparação se for mais parecida com a referência do que 90% dos pares do conjunto.


In [4]:
S = E @ E.T
np.fill_diagonal(S, -1)
iu = np.triu_indices(n, 1)
all_pairs = S[iu]
summary = pd.Series(all_pairs).describe(percentiles=[.1, .25, .5, .75, .9, .99]).round(3)
display(summary.to_frame("todos os pares").T)
FLOOR = float(np.percentile(all_pairs, FLOOR_PERCENTILE))
P90 = float(np.percentile(all_pairs, 90))
print(f"rede · limiar {FLOOR:.3f} (percentil {FLOOR_PERCENTILE})")
above = (S >= P90).sum(axis=1)
print(f"Q2 · limiar {P90:.3f}: mediana de {np.median(above):.0f} disciplinas por referência, "
      f"{int((above < 5).sum())} referências têm menos de cinco")

top = np.argsort(-all_pairs)[:8]
pd.DataFrame([{"a": subjects.at[iu[0][k], "name"], "b": subjects.at[iu[1][k], "name"], "cosseno": round(float(all_pairs[k]), 3)} for k in top])


,count,mean,std,min,10%,25%,50%,75%,90%,99%,max
todos os pares,139128.0,0.8,0.03,0.68,0.763,0.781,0.799,0.818,0.836,0.878,1.0


rede · limiar 0.828 (percentil 85)
Q2 · limiar 0.836: mediana de 46 disciplinas por referência, 1 referências têm menos de cinco


,a,b,cosseno
0,Tópicos Especiais em Estatística Aplicada I,Tópicos Especiais em Estatística Aplicada II,1.0
1,Atividades Teórico-Práticas de Aprofundamento - I,Atividades Teórico-práticas de Aprofundamento II,1.0
2,Tutoria Acadêmica I,Tutoria Acadêmica II,1.0
3,Tópicos Avançados em Ciências de Computação I,Tópicos Avançados em Ciências de Computação II,1.0
4,Prática em Ciência de Dados III,Prática em Ciência de Dados II,1.0
5,Prática em Ciência de Dados I,Prática em Ciência de Dados II,1.0
6,Atividades Acadêmicas de Extensão I,Atividades Acadêmicas de Extensão II,1.0
7,Prática em Ciência de Dados III,Prática em Ciência de Dados I,1.0


## **4. Tópicos com BERTopic**

O agrupamento segue quatro passos.

| passo | método | função |
|---:|---|---|
| 1 | UMAP | reduzir os embeddings a 5 dimensões |
| 2 | HDBSCAN | encontrar grupos com pelo menos 8 disciplinas |
| 3 | c-TF-IDF | destacar palavras específicas de cada grupo |
| 4 | MMR | reduzir palavras redundantes |

Os documentos são os textos de ementa, objetivos e conteúdo programático, sem o nome. Palavras vazias e termos genéricos de ementa são removidos. Itens sem grupo recebem o tópico de centróide mais próximo.

In [5]:
STOP = sorted(set(unicodedata.normalize("NFC", """
a à ao aos as às o os um uma uns umas de da do das dos dum duma em na no nas nos num numa por pela pelo pelas pelos
para pra com sem sob sobre entre até após contra desde perante e ou nem mas porém que se como quando onde quanto
qual quais cujo cuja mais menos muito muitos muita muitas pouco pouca tão tanto também já ainda sempre não sim
seu sua seus suas este esta estes estas esse essa esses essas isso isto aquele aquela aqueles aquelas
ele ela eles elas lhe lhes cada outro outra outros outras mesmo mesma mesmos mesmas todo toda todos todas tal tais
ser é são foi era será serão sendo sido estar está estão ter tem têm haver há pode podem deve devem fazer feito
através via além bem assim etc i ii iii iv v vi
disciplina disciplinas curso cursos aula aulas aluno alunos aluna estudante estudantes professor objetivo objetivos
conceito conceitos básico básicos básica básicas introdução introdutório introdutória fundamentos fundamental
principais principal diversos diversas noções noção visão geral estudo estudos apresentar apresentação abordar
abordagem uso utilização utilizando usando forma formas tipo tipos aspecto aspectos tópico tópicos tema temas
desenvolvimento desenvolver capacidade capaz habilidades habilidade conhecimento conhecimentos compreender
entender permitir oferecer fornecer área áreas parte partes caso casos exemplo exemplos nível níveis
seguintes diferentes relacionados relacionadas importância dentro sistema
""").split()))

docs = subjects["body_all"].tolist()
vectorizer = CountVectorizer(ngram_range=(1, 2), min_df=2, stop_words=STOP, token_pattern=r"(?u)\b[^\W\d_]{3,}\b")
mmr = MaximalMarginalRelevance(diversity=0.3, top_n_words=30)
topic_model = BERTopic(
    umap_model=UMAP(n_neighbors=12, n_components=5, min_dist=0.0, metric="cosine", random_state=SEED),
    hdbscan_model=HDBSCAN(min_cluster_size=MIN_TOPIC, min_samples=3, prediction_data=True),
    vectorizer_model=vectorizer, representation_model=mmr, top_n_words=30,
    language="multilingual", calculate_probabilities=False, verbose=False,
)
raw_topics, _ = topic_model.fit_transform(docs, E)
logging.getLogger("BERTopic").setLevel(logging.ERROR)
noise = int(sum(t == -1 for t in raw_topics))
raw_topics = topic_model.reduce_outliers(docs, raw_topics, strategy="embeddings", embeddings=E)
topic_model.update_topics(docs, topics=raw_topics, vectorizer_model=vectorizer, representation_model=mmr, top_n_words=30)

order = [t for t, _ in Counter(raw_topics).most_common()]
topic = np.array([order.index(t) for t in raw_topics])
T = len(order)
words = [[(w, float(v)) for w, v in topic_model.get_topic(t)][:30] for t in order]
keys = [" ".join(w for w, _ in words[t][:2]) for t in range(T)]
print(f"{T} tópicos, {noise} disciplinas sem tópico no HDBSCAN foram reatribuídas")
pd.DataFrame({"tópico": range(T), "chave": keys, "disciplinas": [int((topic == t).sum()) for t in range(T)],
              "palavras": [", ".join(w for w, _ in words[t][:8]) for t in range(T)],
              "exemplos": [" | ".join(subjects.loc[topic == t, "name"].head(4)) for t in range(T)]})

26 tópicos, 57 disciplinas sem tópico no HDBSCAN foram reatribuídas


,tópico,chave,disciplinas,palavras,exemplos
0,0,ensino educação,69,"ensino, educação, matemática, aprendizagem, at...",Anatomia e Fisiologia para o Engenheiro Bioméd...
1,1,sistemas redes,48,"sistemas, redes, dados, computação, aplicações...",Introdução ao Desenvolvimento Web | Laboratóri...
2,2,teorema funções,33,"teorema, funções, integrais, integral, séries,...",Geometria | Estruturas Algébricas | Desenho Ge...
3,3,modelos testes,30,"modelos, testes, regressão, probabilidade, est...",Controle Adaptativo | Tópicos de Probabilidade...
4,4,quântica onda,27,"quântica, onda, ondas, difração, óptica, óptic...",Química Quântica | Física IV | Introdução à Fí...
5,5,algoritmos programação,27,"algoritmos, programação, árvores, programas, l...",Introdução à Ciência de Computação II | Algori...
6,6,desenho processos,20,"desenho, processos, materiais, calor, manufatu...",Princípios de Aviônica e Navegação | Termodinâ...
7,7,projeto empresa,20,"projeto, empresa, pesquisa, computação, docent...",Metodologia de Pesquisa em Computação | Projet...
8,8,dados visualização,18,"dados, visualização, mineração, mineração dado...",Fundamentos em Ciências de Dados | Análise de ...
9,9,equações lineares,18,"equações, lineares, equações diferenciais, dif...",Matemática Aplicada à Química | Introdução à F...


## **5. Nomes dos tópicos e dicionário de palavras**

O BERTopic entrega palavras, não nomes. Cada tópico recebe um nome curto a partir das palavras de maior peso e das disciplinas do grupo. `data/processed/topic_dictionary.csv` preserva palavras, pesos e posições.

A ordem dos tópicos na visualização (e, com ela, a ordem das cores) segue as folhas de um dendrograma de ligação média sobre os centróides dos tópicos: tópicos parecidos ficam lado a lado e recebem matizes vizinhos.

In [6]:
TOPIC_NAMES = {
    "ensino educação": "Ensino e educação",
    "sistemas redes": "Sistemas computacionais e redes",
    "teorema funções": "Cálculo e análise",
    "modelos testes": "Probabilidade e estatística",
    "quântica onda": "Física quântica, ondas e óptica",
    "algoritmos programação": "Algoritmos e programação",
    "desenho processos": "Materiais, desenho técnico e produção",
    "projeto empresa": "Estágios, projetos e pesquisa",
    "dados visualização": "Mineração e visualização de dados",
    "equações lineares": "Álgebra linear e equações diferenciais",
    "software objetos": "Engenharia de software",
    "extensão atividades": "Extensão e atividades complementares",
    "circuitos transistores": "Eletrônica e sistemas digitais",
    "computação ética": "Ética, tutoria e formação profissional",
    "química reações": "Química",
    "movimento momento": "Mecânica e física geral",
    "imagens processamento": "Processamento de imagens e visão",
    "robôs controle": "Robótica e controle",
    "aprendizado aprendizado máquina": "Aprendizado de máquina",
    "método métodos": "Métodos numéricos",
    "informação sistemas informação": "Gestão, economia e sistemas de informação",
    "embarcados sistemas embarcados": "Sistemas embarcados e microcontroladores",
    "ciência dados ciência": "Práticas em ciência de dados",
    "corrente circuitos": "Eletricidade e circuitos",
    "sistemas sistemas dinâmicos": "Sinais e sistemas de controle",
    "otimização problemas": "Otimização e pesquisa operacional"
}

topic_name = [TOPIC_NAMES.get(k, " · ".join(w for w, _ in words[t][:3])) for t, k in enumerate(keys)]
C = np.array([E[topic == t].mean(axis=0) for t in range(T)])
C /= np.linalg.norm(C, axis=1, keepdims=True)
topic_order = [int(t) for t in leaves_list(linkage(pdist(C, "cosine"), "average", optimal_ordering=True))]
missing = [k for k in keys if k not in TOPIC_NAMES]
print("tópicos sem nome:", missing or "nenhum")

dictionary = pd.DataFrame([{"topic": t, "topic_name": topic_name[t], "rank": r + 1,
                            "word": w, "weight": round(v, 4)} for t in range(T) for r, (w, v) in enumerate(words[t])])
(dictionary[dictionary["rank"] <= 10].groupby(["topic", "topic_name"])["word"].apply(", ".join)
 .reset_index().rename(columns={"word": "palavras de maior peso (c-TF-IDF)"}))

tópicos sem nome: nenhum


,topic,topic_name,palavras de maior peso (c-TF-IDF)
0,0,Ensino e educação,"ensino, educação, matemática, aprendizagem, at..."
1,1,Sistemas computacionais e redes,"sistemas, redes, dados, computação, aplicações..."
2,2,Cálculo e análise,"teorema, funções, integrais, integral, séries,..."
3,3,Probabilidade e estatística,"modelos, testes, regressão, probabilidade, est..."
4,4,"Física quântica, ondas e óptica","quântica, onda, ondas, difração, óptica, óptic..."
5,5,Algoritmos e programação,"algoritmos, programação, árvores, programas, l..."
6,6,"Materiais, desenho técnico e produção","desenho, processos, materiais, calor, manufatu..."
7,7,"Estágios, projetos e pesquisa","projeto, empresa, pesquisa, computação, docent..."
8,8,Mineração e visualização de dados,"dados, visualização, mineração, mineração dado..."
9,9,Álgebra linear e equações diferenciais,"equações, lineares, equações diferenciais, dif..."


## **6. Rede e mapa**

Cada disciplina liga-se às quatro vizinhas com $s(a,b)$ acima do percentil 85, e a vizinha mais próxima é mantida mesmo abaixo do limiar. A posição vem de um UMAP bidimensional e serve para leitura espacial, não como distância exata. Textos idênticos são projetados uma única vez e compartilham a coordenada (`unique=True`). Sem isso, o UMAP os isola como um componente desconexo. As coordenadas são giradas para que o eixo de maior variância fique na horizontal e reescaladas por um único fator, pois normalizar cada eixo separadamente deformaria a projeção. A rede será exportada em `data/processed/similarity_edges.csv`.

In [7]:
edges = set()
for i in range(n):
    for rank, j in enumerate(np.argsort(-S[i])[:K]):
        if rank == 0 or S[i, j] >= FLOOR:
            edges.add((min(i, int(j)), max(i, int(j))))
edges = sorted(edges)
xy = UMAP(n_neighbors=12, n_components=2, min_dist=0.25, metric="cosine", unique=True, random_state=SEED).fit_transform(E)
xy = xy - xy.mean(axis=0)
xy = xy @ np.linalg.svd(xy, full_matrices=False)[2].T
xy = (xy - xy.min(axis=0)) / np.ptp(xy, axis=0).max()
degree = Counter([a for a, _ in edges] + [b for _, b in edges])
print(f"{len(edges)} ligações · grau mediano {np.median([degree[i] for i in range(n)]):.0f} · altura/largura do mapa {np.ptp(xy[:, 1]):.2f}")

1430 ligações · grau mediano 5 · altura/largura do mapa 0.57


## **7. Medidas para as perguntas**

As três perguntas usam os mesmos identificadores de disciplina e tópico.

### **7.1. Q1 · Tópicos e departamentos**

A informação mútua normalizada resume a associação entre tópico $T$ e departamento $D$: $\operatorname{NMI}(T,D)\in[0,1]$. Zero indica independência, e um indica partições idênticas. Como a NMI cresce com o número de grupos mesmo ao acaso, também é informada a versão ajustada (AMI), cujo valor esperado sob independência é zero. O departamento é a sigla do código representante, e códigos numéricos recebem a sigla da unidade.

In [8]:
prefix = subjects["prefix"]
nmi = normalized_mutual_info_score(prefix, topic)
ami = adjusted_mutual_info_score(prefix, topic)
print(f"Q1 · informação mútua tópico × departamento: NMI {nmi:.2f} · AMI {ami:.2f}")
display(pd.crosstab(pd.Series([topic_name[t] for t in topic], name="tópico"), prefix.rename("prefixo")))

Q1 · informação mútua tópico × departamento: NMI 0.50 · AMI 0.43


prefixo,EESC,EESC-ICMC,IAU,ICMC,IFSC,IQSC,LCE,RAD,SAA,SCC,SEL,SEM,SEP,SET,SHS,SLC,SMA,SME,SMM,SSC
tópico,,,,,,,,,,,,,,,,,,,,
Algoritmos e programação,0,0,0,0,0,0,0,0,0,20,0,0,0,0,0,0,2,3,0,2
Aprendizado de máquina,0,0,0,0,0,0,0,0,0,11,0,0,0,0,0,0,0,1,0,2
Cálculo e análise,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,5,26,2,0,0
Eletricidade e circuitos,0,0,0,0,1,0,0,0,0,0,7,0,0,0,0,2,0,0,0,0
Eletrônica e sistemas digitais,0,0,0,0,0,0,0,0,0,0,11,0,0,0,0,0,0,0,0,6
Engenharia de software,0,0,0,0,0,0,0,0,0,1,0,0,1,0,0,0,0,0,0,16
Ensino e educação,1,0,3,0,1,0,0,0,0,0,0,0,0,0,1,38,23,1,0,1
"Estágios, projetos e pesquisa",0,0,0,0,0,0,0,0,0,7,1,0,0,0,0,0,1,6,0,5
Extensão e atividades complementares,9,0,0,0,0,0,3,0,0,2,0,0,0,0,0,2,1,0,0,0


### **7.2. Q3 · Grades por curso**

`data/raw/curriculum.csv` é reescrito com os identificadores das disciplinas únicas. Em duplicidades, obrigatórias prevalecem sobre optativas e fica o semestre mais cedo. A carga horária é a da linha da grade, pois o mesmo conteúdo pode ter cargas distintas em cada curso.

In [9]:
nn = S.argmax(axis=1)
nn_sim = S[np.arange(n), nn]
curriculum = pd.read_csv(RAW_DATA / "curriculum.csv")
course_names = pd.read_csv(RAW_DATA / "courses.csv").set_index("course_id")["course_name"]
code_to_subject = {c: k for k, codes in enumerate(subjects["codes"]) for c in codes.split()}
curriculum["k"] = curriculum["code"].map(code_to_subject)
curriculum = curriculum.dropna(subset=["k"]).astype({"k": int})
KIND = {"obrigatória": 0, "optativa eletiva": 1, "optativa livre": 2}
grade = (curriculum.assign(kind_rank=curriculum["kind"].map(KIND).fillna(2))
         .sort_values(["course_id", "k", "kind_rank", "semester"])
         .drop_duplicates(["course_id", "k"]))
grade = grade[grade.groupby("course_id")["k"].transform("size") >= 10].copy()
grade["mandatory"] = grade["kind_rank"].eq(0)
grade["hours"] = grade["workload_hours"].fillna(0)
grade["topic"] = topic[grade["k"]]
print(f"{grade['course_id'].nunique()} cursos, {len(grade)} disciplinas nas grades "
      f"({int(grade['mandatory'].sum())} obrigatórias)")
grade.groupby("course_id").agg(disciplinas=("k", "size"), obrigatórias=("mandatory", "sum"), semestres=("semester", "max"))

12 cursos, 1061 disciplinas nas grades (502 obrigatórias)


,disciplinas,obrigatórias,semestres
course_id,,,
55030-200,84,26,8
55030-300,78,32,8
55041-0,136,45,10
55051-4,83,43,8
55060-0,82,30,8
55071-4,75,32,8
55090-0,104,30,8
90011-104,44,40,8
90011-204,59,55,8


### **7.3. Q3 · Rede de requisitos**

Os requisitos são filtrados por curso e pelas disciplinas presentes na respectiva grade. Para Matemática, bacharelado e licenciatura são separados. Indicações de conjunto não entram no grafo: pedem que duas disciplinas sejam cursadas juntas e não estabelecem precedência. A saída mede pares, cadeia mais longa e alcance de cada requisito. A semelhança dos pares é comparada com todos os pares e com os pares de disciplinas da mesma grade.

In [10]:
requisites_all = pd.read_csv(RAW_DATA / "requisites.csv", dtype={"course_code": str})
HABILITATION = {"55030-200": "Bacharelado", "55030-300": "Licenciatura"}
course_reqs = []
for cid, g in grade.groupby("course_id"):
    rows = requisites_all[(requisites_all["course_code"] == cid.split("-")[0]) & (requisites_all["relation"] != "indicação de conjunto")]
    if cid in HABILITATION:
        rows = rows[rows["course_name"].str.contains(HABILITATION[cid])]
    in_grade = set(g["k"])
    for r in rows.itertuples():
        a, b = code_to_subject.get(r.requisite), code_to_subject.get(r.code)
        if a is None or b is None or a == b or a not in in_grade or b not in in_grade:
            continue
        course_reqs.append({"course_id": cid, "a": a, "b": b, "strong": r.relation == "requisito"})
course_reqs = (pd.DataFrame(course_reqs).groupby(["course_id", "a", "b"])["strong"].any().reset_index())
course_reqs["similarity"] = [round(float(S[a, b]), 3) for a, b in zip(course_reqs["a"], course_reqs["b"])]

summary = []
for cid, g in course_reqs.groupby("course_id"):
    G = nx.DiGraph(list(zip(g["a"], g["b"])))
    for c in list(nx.simple_cycles(G)):
        if G.has_edge(c[-1], c[0]):
            G.remove_edge(c[-1], c[0])
    longest = nx.dag_longest_path(G)
    hub = max(G.nodes, key=lambda v: len(nx.descendants(G, v)))
    summary.append({"curso": course_names.get(cid, cid), "pares": len(g), "disciplinas com requisito": g["b"].nunique(),
                    "cadeia mais longa (etapas)": len(longest) - 1,
                    "cadeia": " → ".join(subjects.at[v, "name"] for v in longest),
                    "mais disciplinas dependem de": f"{subjects.at[hub, 'name']} ({len(nx.descendants(G, hub))})"})
same_grade = np.concatenate([S[np.ix_(k, k)][np.triu_indices(len(k), 1)] for k in (g["k"].to_numpy() for _, g in grade.groupby("course_id"))])
distinct = course_reqs.drop_duplicates(["a", "b"])["similarity"]
print(f"Q3 · {len(course_reqs)} pares requisito → disciplina dentro das grades ({len(distinct)} distintos) · semelhança mediana "
      f"{course_reqs['similarity'].median():.2f} (distintos {distinct.median():.2f}) vs. mesma grade {np.median(same_grade):.2f} "
      f"e todos os pares {np.median(all_pairs):.2f}")
pd.DataFrame(summary)

Q3 · 897 pares requisito → disciplina dentro das grades (576 distintos) · semelhança mediana 0.85 (distintos 0.85) vs. mesma grade 0.81 e todos os pares 0.80


,curso,pares,disciplinas com requisito,cadeia mais longa (etapas),cadeia,mais disciplinas dependem de
0,Bacharelado em Matemática,76,56,4,Introdução à Programação de Computadores → Int...,Cálculo I (20)
1,Licenciatura em Matemática,62,45,4,Cálculo I → Cálculo II → Espaços Métricos → To...,Cálculo I (19)
2,Bacharelado em Ciências de Computação,107,88,6,Introdução à Lógica Digital → Sistemas Digitai...,Algoritmos e Estruturas de Dados I (51)
3,Bacharelado em Sistemas de Informação,63,54,5,Introdução à Ciência de Computação I → Algorit...,Introdução à Ciência de Computação I (40)
4,Bacharelado em Matemática Aplicada e Computaçã...,77,59,4,Cálculo I → Física I → Física II → Fundamentos...,Cálculo I (31)
5,Bacharelado em Estatística e Ciência de Dados,74,54,4,Introdução à Ciência de Computação I → Estrutu...,Cálculo I (38)
6,Bacharelado em Ciência de Dados,100,79,5,Introdução à Ciência de Computação I → Estrutu...,Introdução à Ciência de Computação I (45)
7,Licenciatura em Ciências Exatas - Núcleo Geral,37,22,5,Elementos de Matemática → Cálculo I → Física I...,Elementos de Matemática (14)
8,Licenciatura em Ciências Exatas - Habilitação ...,63,35,7,Elementos de Matemática → Cálculo I → Física I...,Elementos de Matemática (27)
9,Licenciatura em Ciências Exatas - Habilitação ...,54,37,7,Elementos de Matemática → Cálculo I → Física I...,Química 1 (26)


### **7.4. Carga horária por tópico**

Para cada curso $c$ e tópico $t$, soma-se $H_{c,t}=\sum_{d\in(c,t)}h_d$. A tabela mantém cursos nas colunas e tópicos nas linhas e alimenta a composição exibida no treemap.

In [11]:
course_topic = (grade.assign(course=grade["course_id"].map(course_names), topic_name=grade["topic"].map(lambda t: topic_name[t]))
                .pivot_table(index="topic_name", columns="course", values="hours", aggfunc="sum", fill_value=0))
print("Horas por tópico em cada curso (todas as disciplinas):")
course_topic.astype(int)

Horas por tópico em cada curso (todas as disciplinas):


course,Bacharelado em Ciência de Dados,Bacharelado em Ciências de Computação,Bacharelado em Estatística e Ciência de Dados,Bacharelado em Matemática,Bacharelado em Matemática Aplicada e Computação Científica,Bacharelado em Sistemas de Informação,Engenharia de Computação,Licenciatura em Ciências Exatas - Habilitação em Física,Licenciatura em Ciências Exatas - Habilitação em Matemática,Licenciatura em Ciências Exatas - Habilitação em Química,Licenciatura em Ciências Exatas - Núcleo Geral,Licenciatura em Matemática
topic_name,,,,,,,,,,,,
Algoritmos e programação,825,1515,255,885,645,630,960,0,0,0,0,480
Aprendizado de máquina,570,705,300,270,270,420,480,0,0,0,0,90
Cálculo e análise,420,300,540,1590,870,90,300,180,600,180,180,1770
Eletricidade e circuitos,0,0,0,60,60,0,180,150,0,0,0,60
Eletrônica e sistemas digitais,90,330,0,0,0,60,450,0,0,0,0,0
Engenharia de software,405,795,0,120,120,615,660,0,0,0,0,0
Ensino e educação,60,75,600,1305,60,0,135,2550,2400,2490,1920,2655
"Estágios, projetos e pesquisa",1260,2970,900,0,720,2880,1650,0,0,0,0,90
Extensão e atividades complementares,0,150,0,30,0,0,180,0,0,0,0,0


## **8. Exportação**

A etapa final grava as medidas em `data/processed/` e reúne o conteúdo consumido pelo site em `html/data/disciplinas.js`.

| arquivo | conteúdo |
|---|---|
| `subject_topics.csv` | tópico, posição e vizinha de cada disciplina |
| `topics.csv`, `topic_dictionary.csv` | nomes, tamanhos e palavras |
| `similarity_edges.csv` | rede de semelhança |
| `course_requisites.csv` | requisitos por curso |
| `course_topic_hours.csv` | carga por tópico e curso |
| `html/data/disciplinas.js` | pacote completo da visualização |

In [12]:
nodes = []
for k, row in subjects.iterrows():
    nodes.append({
        "id": row["subject_id"], "codes": row["codes"].split(), "name": row["name"], "dept": prefix[k],
        "topic": int(topic[k]),
        "x": round(float(xy[k, 0]), 4), "y": round(float(xy[k, 1]), 4),
        "nn": int(nn[k]), "nnSim": round(float(nn_sim[k]), 3),
        "credits": None if pd.isna(row["credits"]) else int(row["credits"]),
        "hours": None if pd.isna(row["workload_hours"]) else int(row["workload_hours"]),
        "period": None if pd.isna(row["ideal_period"]) else int(row["ideal_period"]),
        "courses": int(row["n_courses"]), "teachers": row["teachers"],
        "ementa": row["ementa"], "objectives": row["objectives"], "program": row["program"],
        "near": [[int(j), round(float(S[k, j]), 3)] for j in np.argsort(-S[k])[:8]],
    })
topics_out = [{"id": t, "key": keys[t], "name": topic_name[t], "size": int((topic == t).sum()),
               "words": [[w, round(v, 4)] for w, v in words[t]],
               "depts": dict(Counter(prefix[topic == t]).most_common())} for t in range(T)]
payload = {
    "meta": {"collectedAt": collection["collected_at_utc"], "pages": int(collection["disciplines_total"]), "subjects": n,
             "topics": T, "model": MODEL, "k": K, "floor": round(FLOOR, 3), "nmi": round(float(nmi), 3), "ami": round(float(ami), 3),
             "medianAll": round(float(np.median(all_pairs)), 3), "p90All": round(P90, 3), "simScale": 1000},
    "topics": topics_out, "topicOrder": topic_order, "nodes": nodes,
    "edges": [[a, b, round(float(S[a, b]), 3)] for a, b in edges],
    # semelhança de cada disciplina com todas as outras, em milésimos (usada no beeswarm da Q2)
    "sim": [[int(round(max(0.0, float(v)) * 1000)) for v in row] for row in S],
    "courses": [{"id": cid, "name": course_names.get(cid, cid),
                 "items": [[int(r.k), int(r.mandatory), int(r.semester), int(r.kind_rank)] for r in g.itertuples()],
                 "reqs": [[int(r.a), int(r.b), int(r.strong)] for r in course_reqs[course_reqs["course_id"] == cid].itertuples()]}
                for cid, g in grade.groupby("course_id")],
}
SITE_DATA.mkdir(parents=True, exist_ok=True)
(SITE_DATA / "disciplinas.js").write_text("window.DATA=" + json.dumps(payload, ensure_ascii=False, separators=(",", ":")) + ";\n", encoding="utf-8")

outputs = {
    "subject_topics.csv": pd.DataFrame({"subject_id": subjects["subject_id"], "name": subjects["name"], "topic": topic,
                                        "topic_name": [topic_name[t] for t in topic],
                                        "x": xy[:, 0].round(4), "y": xy[:, 1].round(4),
                                        "nearest": subjects["subject_id"].to_numpy()[nn], "nearest_similarity": nn_sim.round(3)}),
    "topics.csv": pd.DataFrame([{"topic": t["id"], "name": t["name"], "size": t["size"],
                                 "words": ", ".join(w for w, _ in t["words"][:15])} for t in topics_out]),
    "topic_dictionary.csv": dictionary,
    "similarity_edges.csv": pd.DataFrame([{"source": subjects.at[a, "subject_id"], "target": subjects.at[b, "subject_id"],
                                           "similarity": round(float(S[a, b]), 3)} for a, b in edges]),
    "course_requisites.csv": course_reqs.assign(course=course_reqs["course_id"].map(course_names),
                                                requisite=subjects["subject_id"].to_numpy()[course_reqs["a"]],
                                                discipline=subjects["subject_id"].to_numpy()[course_reqs["b"]]).drop(columns=["a", "b"]),
    "course_topic_hours.csv": course_topic.reset_index(),
}
for name, frame in outputs.items():
    frame.to_csv(PROCESSED / name, index=False)
artifacts = [{"artifact": name, "rows": len(frame), "directory": "data/processed"} for name, frame in outputs.items()]
artifacts.append({"artifact": "disciplinas.js", "rows": n, "directory": "html/data"})
pd.DataFrame(artifacts)

,artifact,rows,directory
0,subject_topics.csv,528,data/processed
1,topics.csv,26,data/processed
2,topic_dictionary.csv,780,data/processed
3,similarity_edges.csv,1430,data/processed
4,course_requisites.csv,897,data/processed
5,course_topic_hours.csv,26,data/processed
6,disciplinas.js,528,html/data
